# 22. PDF 원문 발췌 답변과 출처

[프로젝트 적용] 21번의 전체 DB 검색을 재사용합니다. 문서 벡터는 다시 계산하지 않습니다.
[프로젝트 추가] 선택한 방식은 **설명서의 검색용 글을 발췌**하는 것입니다. 새 문장을 만드는 LLM은 아직 연결하지 않습니다.

확인한 근거와 필수 각주를 함께 보여주고, 사용할 수 있는 그림만 연결합니다. 미검토 자료만 찾았으면 검토 필요, 근거를 고르지 못했으면 확인 불가로 안내합니다. 단어 일치는 임시 규칙이므로 질문의 모든 조건을 만족하는지 의미 검증한 것은 아닙니다.


In [ ]:
from pathlib import Path
import json
import sys

# 실행 폴더에 따라 PDF 경로가 달라졌던 오류를 피하도록 프로젝트 루트를 찾습니다.
project = Path.cwd().resolve()
if project.name == "notebooks":
    project = project.parent
if not (project / "src/car_search_rag/zzong_santafe_lag").is_dir():
    raise FileNotFoundError("프로젝트 폴더 또는 notebooks 폴더에서 실행하세요.")
if str(project / "src") not in sys.path:
    sys.path.insert(0, str(project / "src"))


In [ ]:
from car_search_rag.zzong_santafe_lag.answer_service import ManualAnswerService

# 실제 전체 저장을 완료한 작업 번호를 결과 기록에서 읽습니다.
report_path = project / "data/zzong_santafe_lag/reports/full_store_20261002.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
if report["phase"] != "completed":
    raise ValueError("전체 DB 저장 완료를 먼저 확인하세요.")
answer_service = ManualAnswerService(report["save_result"]["run_id"])
print("발췌 답변 준비 완료. 모델은 첫 질문에서 준비합니다.")


In [ ]:
# [프로젝트 적용] 이 셀의 질문을 바꿔 실행합니다. 같은 서비스의 로컬 모델을 재사용합니다.
question = "차대번호는 어디에서 확인하나요?"
answer = answer_service.answer(question, top_k=3, progress=print)
print(answer["answer"])
if answer["status"] != "evidence_excerpt":
    print("이유:", answer["reason"])


In [ ]:
# [프로젝트 추가] 발췌와 출처를 따로 보관하므로 나중에 화면에서도 연결할 수 있습니다.
for source in answer["sources"]:
    print(source["label"], "| 제목:", source["title"])
    print("검토 상태:", source["verification_status"])
print("LLM 호출 여부:", answer["llm_called"])
print("DB 변경 여부:", answer["db_written"])
print("답변 상태:", answer["status"])


In [ ]:
from IPython.display import Image, display

# 공개 파일이 있고 설명·문맥 연결을 확인한 그림만 표시합니다.
for image in answer["images"]:
    print("PDF 쪽:", image["pdf_page_number"], "| 파일:", image["file_name"])
    for description in image["descriptions"]:
        print(description["description"])
    display(Image(url=image["public_url"]))
print("파일 업로드 또는 문맥 연결 확인 전인 참조:", len(answer["pending_image_references"]))


## 답변 상태 읽기

- `evidence_excerpt`: 확인한 검색용 글을 발췌했습니다. 모든 질문 조건을 검증한 생성 답변은 아닙니다.
- `needs_review`: 관련 후보는 있으나 원문 또는 이어지는 주의사항의 확인이 필요합니다.
- `insufficient_evidence`: 현재 후보에서 관련 있는 확인 자료를 선택하지 못했습니다. PDF 전체에 없다고 판정한 것은 아닙니다.
- `outside_pdf_scope`: 최신 정보·개별 차량 실제 고장 여부처럼 현재 PDF로 판단할 수 없는 명시적 요청입니다.

원문(raw_text)은 대조용으로 보관하고 화면에는 전처리·표 정리를 적용한 content를 발췌합니다. 그림을 직접 분석하는 모델·OCR은 아직 없습니다.
